# 09 — Handling Long Reports

**Goal:** Real CTI reports routinely run 2,000–10,000 tokens. BERT only handles 512. You'll learn three ways to deal with this:

1. **Chunking with overlap** (sliding window) — the workhorse
2. **Aggregation** — combining per-chunk predictions into a single report-level answer
3. **Long-context alternatives** — when to reach for Longformer / BigBird

We'll use the classifier from notebook 8, but the same pattern works for NER (chunk, predict, merge spans).

## Step 1 — Build a long synthetic report

In [1]:
long_report = """
Threat actor group APT29, also known as Cozy Bear, has been active since at least 2008.
Our analysis of recent activity shows targeting of diplomatic entities across Europe.
Initial access was achieved through spear-phishing emails containing malicious ISO attachments.
Once opened, the ISO dropped a custom loader that retrieved a second-stage payload from cloud storage.
Persistence was established via scheduled tasks and registry Run keys.
Lateral movement relied on stolen credentials harvested with a modified version of Mimikatz.
The group exfiltrated data through legitimate cloud services, making detection difficult.
In one incident, over 40 gigabytes of internal communications were stolen over six weeks.
The campaign demonstrates continued investment in stealthy, long-dwell operations.
Indicators of compromise include specific C2 domains and a small set of file hashes, listed below.
Affected organizations should review authentication logs and cloud egress traffic for anomalies.
Patch levels for Outlook and VPN appliances should be verified, given recent exploitation trends.
""" * 10  # repeat to make it genuinely long

print("Words:", len(long_report.split()))

Words: 1540


## Step 2 — Confirm it exceeds 512 tokens

In [2]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("./cls-cti-bert-final")

tokens = tokenizer.tokenize(long_report)
print("Total BERT tokens:", len(tokens))

Token indices sequence length is longer than the specified maximum sequence length for this model (1980 > 512). Running this sequence through the model will result in indexing errors


Total BERT tokens: 1980


## Step 3 — Chunk with overlap

The tokenizer supports this directly via `return_overflowing_tokens=True` + `stride`.

- `max_length` — window size
- `stride` — how much the next window overlaps the previous one

In [3]:
enc = tokenizer(
    long_report,
    max_length=512,
    stride=64,
    truncation=True,
    return_overflowing_tokens=True,
    return_tensors="pt",
    padding=True,
)

n_chunks = enc["input_ids"].shape[0]
print(f"Report was split into {n_chunks} chunks of length {enc['input_ids'].shape[1]}")

Report was split into 5 chunks of length 512


## Step 4 — Classify each chunk, then aggregate

For classification there are two reasonable aggregation strategies:

- **Majority vote** — whichever class wins most chunks
- **Average softmax** — average the per-class probabilities across chunks, then argmax

Average-softmax is usually the more stable of the two because it's sensitive to confidence.

In [4]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("./cls-cti-bert-final")
model.eval()
id2label = model.config.id2label

with torch.no_grad():
    logits = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).logits

probs = F.softmax(logits, dim=-1)            # [n_chunks, n_classes]
per_chunk_preds = probs.argmax(-1).tolist()

print("Per-chunk predictions:")
for i, p in enumerate(per_chunk_preds):
    print(f"  chunk {i}: {id2label[p]}")

avg_probs = probs.mean(dim=0)                 # [n_classes]
final_label = id2label[avg_probs.argmax().item()]
print(f"\nAggregated label (avg softmax): {final_label}")
print("Class probabilities:")
for i, p in enumerate(avg_probs.tolist()):
    print(f"  {id2label[i]:<24} {p:.3f}")

Per-chunk predictions:
  chunk 0: phishing
  chunk 1: phishing
  chunk 2: phishing
  chunk 3: phishing
  chunk 4: ransomware

Aggregated label (avg softmax): phishing
Class probabilities:
  apt-campaign             0.173
  phishing                 0.247
  ransomware               0.209
  supply-chain             0.143
  vulnerability-report     0.228


## Step 5 — NER on long documents

For NER the pattern is the same (chunk with overlap), but aggregation is span-level:

1. Tokenize with `return_overflowing_tokens=True` and keep the `offset_mapping`.
2. Run the NER model on each chunk.
3. Map each predicted entity back to character offsets in the **original** document.
4. Deduplicate entities found in the overlap region.

In practice, `transformers.pipeline("token-classification", aggregation_strategy="simple")` plus the `chunk_length` argument handles this. For custom pipelines you implement it yourself.

Skeleton:

```python
enc = tokenizer(doc, return_offsets_mapping=True, return_overflowing_tokens=True,
                max_length=512, stride=64, truncation=True)
# for each chunk: predict labels, use offset_mapping to map back to doc
# dedupe spans whose character ranges overlap between adjacent chunks
```

(Full implementation is one of the exercises.)

## Step 6 — When to use long-context models instead

| Model | Max tokens | When it's worth switching |
|---|---|---|
| BERT / RoBERTa | 512 | Default |
| DeBERTa-v3 | 512–1024 | Small quality bump |
| Longformer | 4,096 | Long CTI docs, global attention tokens |
| BigBird | 4,096 | Similar to Longformer |
| ModernBERT | 8,192 | Current best for long documents |

Trade-off: long-context models are larger and slower. For most CTI use cases, chunking a 512-token BERT is good enough and lets you reuse your existing fine-tuned models.

## Your turn — exercises

1. **Majority vote** aggregation: implement it and compare against average-softmax on a handful of long reports.
2. **Stride sensitivity**: try `stride = 0, 32, 64, 128, 256`. Plot how the final label changes (or doesn't) — how sensitive is the result?
3. **NER chunking**: implement the NER long-document pipeline described in Step 5. Use the sentence `("APT28 ... " * 50)` repeated, and verify you still get `APT28` entities out.

## Next notebook

**`10_domain_models_securebert.ipynb`** — swap in a CTI-domain BERT (SecureBERT) and compare on the same tasks.